In [ ]:
import sys
from tqdm import tqdm

BASE_PATH = "/work/nvme/bbjs/sbharadwaj/powsm/xeuspr"
sys.path.append(BASE_PATH)
from src.data.kaldi_dataset import build_kaldi_datamodule

datamodule = build_kaldi_datamodule(
    "doreco",
    data_dir="/work/hdd/bbjs/shared/powsm/s2t1/dump/raw",
    dataset_config_path=f"{BASE_PATH}/configs/data/powsm_evalset_index.yaml",
    portable_wavscp=False,
    sampling_rate=16000,
    batch_size=2,
    num_workers=1,
)
datamodule.setup()

key2speech = {}
c = 0
for batch in tqdm(datamodule.predict_dataloader().dataset):
    key = batch["key"]
    speech = batch["speech"].cpu().numpy()
    key2speech[key] = speech
    c += 1
    if c % 100 == 0:
        break

In [ ]:
import json

prediction_path = "/work/nvme/bbjs/sbharadwaj/powsm/xeuspr/exp/runs/inf_doreco_xeuspr/030000/transcription.json"

with open(prediction_path) as f:
    predictions = json.load(f)

pred = {"utt_id": [], "target": [], "prediction": []}
for idx in tqdm(predictions):
    item = predictions[idx]
    pred["utt_id"].append(item["passthrough"]["utt_id"])
    pred["target"].append(item["passthrough"]["target"])
    pred["prediction"].append(item["pred"][0]["processed_transcript"])

del predictions

print(f"loaded {len(pred['utt_id'])} predictions")

In [ ]:
import pandas as pd

PRED = {k: [] for k, _ in pred.items()}
PRED["speech"] = []
for i in tqdm(range(len(pred["utt_id"]))):
    utt_id = pred["utt_id"][i]
    if utt_id in key2speech:
        PRED["speech"].append(key2speech[utt_id])
        PRED["utt_id"].append(utt_id)
        PRED["target"].append(pred["target"][i])
        PRED["prediction"].append(pred["prediction"][i])

# del key2speech
df = pd.DataFrame(PRED)

In [ ]:
import numpy as np
import base64
import io
from scipy.io.wavfile import write
from IPython.display import HTML


# Helper to convert a single list of floats to an HTML player
def audio_tag(data):
    # Ensure it's a numpy array for processing
    audio_arr = np.array(data, dtype=np.float32)
    byte_io = io.BytesIO()
    # Replace 16000 with your actual sample rate
    write(byte_io, 16000, audio_arr)
    b64 = base64.b64encode(byte_io.getvalue()).decode("utf-8")
    return f'<audio controls style="width:120px; height:30px;"><source src="data:audio/wav;base64,{b64}" type="audio/wav"></audio>'


# Run on the first 10 rows
HTML(df.iloc[:10].to_html(escape=False, formatters={"speech": audio_tag}))

In [ ]:
import numpy as np
import panphon
import panphon.distance
from tqdm import tqdm
from tabulate import tabulate
import unicodedata
import string

# 1. Initialize Tools
ft = panphon.FeatureTable()
dst = panphon.distance.Distance()


def get_alignment_path(ref_segs, hyp_segs):
    """Standard Levenshtein DP to get the path."""
    R, H = len(ref_segs), len(hyp_segs)
    d = np.zeros((R + 1, H + 1))
    for i in range(R + 1):
        d[i, 0] = i
    for j in range(H + 1):
        d[0, j] = j

    for i in range(1, R + 1):
        for j in range(1, H + 1):
            cost = 0 if ref_segs[i - 1] == hyp_segs[j - 1] else 1
            d[i, j] = min(d[i - 1, j] + 1, d[i, j - 1] + 1, d[i - 1, j - 1] + cost)

    path = []
    i, j = R, H
    while i > 0 or j > 0:
        if (
            i > 0
            and j > 0
            and d[i, j]
            == d[i - 1, j - 1] + (0 if ref_segs[i - 1] == hyp_segs[j - 1] else 1)
        ):
            path.append(("match/sub", i - 1, j - 1))
            i -= 1
            j -= 1
        elif i > 0 and d[i, j] == d[i - 1, j] + 1:
            path.append(("del", i - 1, None))
            i -= 1
        else:
            path.append(("ins", None, j - 1))
            j -= 1
    return path[::-1]


# Configuration
N_QUANTILES = 10
# We store totals to compute the final averages
totals = [{"fed_sum": 0.0, "ref_phones": 0} for _ in range(N_QUANTILES)]

# 2. Processing Loop
for i in tqdm(range(len(pred["utt_id"])), desc="Evaluating spans"):
    ref_txt = unicodedata.normalize(
        "NFD",
        pred["target"][i]
        .replace(" ", "")
        .translate(str.maketrans("", "", string.punctuation)),
    )
    hyp_txt = unicodedata.normalize(
        "NFD",
        pred["prediction"][i]
        .replace(" ", "")
        .translate(str.maketrans("", "", string.punctuation)),
    )

    ref_segs = ft.ipa_segs(ref_txt)
    hyp_segs = ft.ipa_segs(hyp_txt)
    n_ref = len(ref_segs)

    if n_ref < N_QUANTILES:
        continue  # Ensure enough segments to split

    path = get_alignment_path(ref_segs, hyp_segs)

    # Calculate quantile boundaries for Ground Truth
    # Indices for each quantile: [q_start, q_end)
    for q_idx in range(N_QUANTILES):
        q_start = (q_idx * n_ref) // N_QUANTILES
        q_end = ((q_idx + 1) * n_ref) // N_QUANTILES

        # 3. Find aligned span in predicted string
        # We look for all entries in the path where the reference index falls in [q_start, q_end)
        # We also include any 'ins' ops that occur immediately before or during this GT span.

        # Find path segment indices
        path_indices = []
        for idx, (op, r_i, h_i) in enumerate(path):
            # If the path entry involves a reference phone within our current quantile
            if r_i is not None and q_start <= r_i < q_end:
                path_indices.append(idx)

        if not path_indices:
            continue

        # Extract the sequence of predicted segments (hyp_segs) aligned to this path segment
        p_start_in_path = min(path_indices)
        p_end_in_path = max(path_indices)

        # Get all hyp indices within the range of the path corresponding to this GT quantile
        sub_hyp_indices = [
            path[k][2]
            for k in range(p_start_in_path, p_end_in_path + 1)
            if path[k][2] is not None
        ]

        # Prepare substrings
        gt_span_str = "".join(ref_segs[q_start:q_end])

        if sub_hyp_indices:
            h_min, h_max = min(sub_hyp_indices), max(sub_hyp_indices)
            hyp_span_str = "".join(hyp_segs[h_min : h_max + 1])
        else:
            hyp_span_str = ""

        # 4. Call FED distance error once for this full span
        # feature_edit_distance returns total distance for the string pair
        fed_val = dst.feature_edit_distance(gt_span_str, hyp_span_str)

        # 5. Store for PFER (fed / number of GT phones)
        totals[q_idx]["fed_sum"] += fed_val
        totals[q_idx]["ref_phones"] += q_end - q_start

# 4. Generate Report
table_data = []
for idx, t in enumerate(totals):
    n = t["ref_phones"]
    pfer = (t["fed_sum"] / n) if n > 0 else 0
    table_data.append(
        [
            f"Part {idx+1} ({(idx*100//N_QUANTILES)}%-{((idx+1)*100//N_QUANTILES)}%)",
            f"{pfer:.4f}",
            n,
        ]
    )

print("\n" + "=" * 60)
print("SPAN-BASED HYPOTHESIS TEST: FED DISTRIBUTION")
print("=" * 60)
print(tabulate(table_data, headers=["Portion", "Avg PFER (FED/N)", "Ref Phones"]))

In [ ]:
from collections import Counter
import pandas as pd

ft = panphon.FeatureTable()
phone_errors = Counter()
attr_errors = Counter()

for i in tqdm(range(len(pred["utt_id"])), desc="Analyzing Errors"):
    ref_segs = ft.ipa_segs(
        unicodedata.normalize("NFD", pred["target"][i].replace(" ", ""))
    )
    hyp_segs = ft.ipa_segs(
        unicodedata.normalize("NFD", pred["prediction"][i].replace(" ", ""))
    )

    path = get_alignment_path(ref_segs, hyp_segs)

    for op, r_idx, h_idx in path:
        if op == "match/sub" and ref_segs[r_idx] != hyp_segs[h_idx]:
            # Top mistaken phones
            phone_errors[f"{ref_segs[r_idx]} → {hyp_segs[h_idx]}"] += 1

            # Feature attribute errors
            r_vec = ft.word_to_vector_list(ref_segs[r_idx])[0]
            h_vec = ft.word_to_vector_list(hyp_segs[h_idx])[0]
            for val_r, val_h, name in zip(r_vec, h_vec, ft.names):
                if val_r != val_h:
                    attr_errors[name] += 1

        elif op == "del":
            phone_errors[f"{ref_segs[r_idx]} → [DEL]"] += 1
        elif op == "ins":
            phone_errors[f"[INS] → {hyp_segs[h_idx]}"] += 1

# Display Results
print("\n### TOP 10 PHONE MISTAKES")
print(pd.Series(dict(phone_errors.most_common(10))).to_string())

print("\n### TOP 10 MISTAKEN ATTRIBUTES (Feature Errors)")
print(pd.Series(dict(attr_errors.most_common(10))).to_string())

# Add this at the end of your analysis script
total_subs = sum(
    1 for op, r, h in path if op == "match/sub" and ref_segs[r] != hyp_segs[h]
)

print(f"\n### FEATURE ERROR DENSITY (Errors per Substitution)")
for attr, count in attr_errors.most_common(10):
    # This shows what % of mistakes involve this specific feature
    percentage = (count / total_subs) * 100 if total_subs > 0 else 0
    print(f"{attr:10} : {percentage:5.1f}% of substitutions")

# PR loss analysis

In [1]:
import sys
from tqdm import tqdm

BASE_PATH = "/work/nvme/bbjs/sbharadwaj/powsm/xeuspr"

sys.path.append(BASE_PATH)
from src.data.kaldi_pretraining_dataset import build_kaldi_datamodule

datamodule = build_kaldi_datamodule(
    "pr_fixed",
    dataset_config_path=f"{BASE_PATH}/configs/data/ipapack_index.yaml",
    vocab_file=f"{BASE_PATH}/src/model/xeusphoneme/resources/ipa_vocab.json",
    batch_size=1,
    num_workers=30,
    limit_samples=None,
    filter_langs=["eng"],
    read_asr_text=True,
)
datamodule.setup()

print("Loaded dataset with length:", len(datamodule.val_dataloader()))

/work/nvme/bbjs/sbharadwaj/powsm/xeuspr/.venv_dai/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Reading text: 18620it [00:00, 482252.76it/s]
Reading language: 18620it [00:00, 504443.49it/s]


READ ASR TEXT: True


Reading ASR text: 18620it [00:00, 634963.25it/s]


Loaded ASR text for 4655 samples
Filtering dataset by languages ['eng']. Reduced samples from 4655 to 263.


Reading text: 18620it [00:00, 501360.58it/s]
Reading language: 18620it [00:00, 512830.56it/s]


...

Loaded ASR text for 4655 samples
Filtering dataset by languages ['eng']. Reduced samples from 4655 to 263.


Reading text: 18620it [00:00, 503182.44it/s]
Reading language: 18620it [00:00, 519164.66it/s]


READ ASR TEXT: True


Reading ASR text: 18620it [00:00, 648180.24it/s]

Loaded ASR text for 4655 samples
Filtering dataset by languages ['eng']. Reduced samples from 4655 to 263.
Loaded dataset with length: 263


In [4]:
import torch
from src.recipe.phone_recognition.model_module import PhoneRecognitionModel
from src.model.xeusphoneme.builders import build_xeus_pr_from_hf
import json

CKPT_PATH = "/work/nvme/bbjs/sbharadwaj/powsm/xeuspr/exp/runs/train_ipapack_xeuspr/20251225_221757/checkpoints/step_030000.ckpt"
# CKPT_PATH = "/work/nvme/bbjs/sbharadwaj/powsm/xeuspr/exp/runs/train_ipapack_xeuspr/20251231_100230/checkpoints/step_246268.ckpt"
VOCAB_FILE = f"{BASE_PATH}/src/model/xeusphoneme/resources/ipa_vocab.json"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device = ", device)

net = build_xeus_pr_from_hf(
    work_dir=f"{BASE_PATH}/exp/cache/xeus",
    checkpoint=CKPT_PATH,
    vocab_file=VOCAB_FILE,
    config_file=None,
    hf_repo="espnet/xeus",
)

model = PhoneRecognitionModel(net=net, optimizer=None)

model.to(device)
model.eval()

with open(VOCAB_FILE, "r") as f:
    vocab = json.load(f)
id2token = {k: v for v, k in vocab.items()}

Returning existing local_dir `/work/nvme/bbjs/sbharadwaj/powsm/xeuspr/exp/cache/xeus` as remote repo cannot be accessed in `snapshot_download` (None).


Using device =  cuda


In [ ]:
def get_phone_str(token_ids):
    return "/".join([id2token[t] for t in token_ids if t in id2token])


results = []
dataloader = datamodule.val_dataloader()
with torch.no_grad():
    ctr = 0
    for batch in tqdm(dataloader, desc="Calculating losses"):
        batch = {
            k: v.to(device) if isinstance(v, torch.Tensor) else v
            for k, v in batch.items()
        }
        out = model(batch)
        # Extract individual losses if your loss function returns a vector
        # If 'out["loss"]' is a scalar (mean), you'll need to modify your model
        # to return per-sample loss for precise filtering.
        # Assuming for now it's a scalar or we process batch by batch:
        batch_loss = out["loss"].item()
        # Store keys and their associated loss
        for i, key in enumerate(batch["keys"]):
            results.append(
                {
                    "key": key,
                    "loss": batch_loss,  # avg
                    "speech": batch["speech"][i].cpu().numpy(),
                    "phone_str": get_phone_str(batch["text"][i].cpu().numpy()),
                    "language": batch["lang_sym"][i],
                    "asr_text": batch["asr_text"],
                }
            )
        ctr += 1
        if ctr > 200:
            break

results.sort(key=lambda x: x["loss"], reverse=True)

Calculating losses:   6%|▌         | 16/263 [00:06<00:27,  8.86it/s]

In [ ]:
threshold = 0
high_loss_samples = [r for r in results if r["loss"] > threshold]

print(f"Found {len(high_loss_samples)} samples with loss > {threshold}")


def display_audio(data):
    from IPython.display import Audio, display

    display(Audio(data, rate=16000))


for sample in high_loss_samples:
    print(f"Key: {sample['key']} | Language: {sample['language']}")
    print(f"Loss: {sample['loss']:.4f} | phone: {sample['phone_str']}")
    print(f"asr: {sample['asr_text']}")
    display_audio(sample["speech"])
    print("===" * 40)

In [ ]:
batch

In [ ]:
import numpy as np

pth = "/work/nvme/bbjs/sbharadwaj/powsm/xeuspr/exp/cache/lengths_pr.npy"
lengths = np.load(pth, allow_pickle=True)
lengths